In [680]:
import pandas as pd
from pathlib import Path

RUTA = Path("../data")

In [681]:
orders = pd.read_csv(
    RUTA / "instacart_orders.csv",
    sep=";"
)

order_prods = pd.read_csv(
    RUTA / "order_products.csv",
    sep=";"
)

products = pd.read_csv(
    RUTA / "products.csv",
    sep=";"
)

aisles = pd.read_csv(
    RUTA / "aisles.csv",
    sep=";"
)

departments = pd.read_csv(
    RUTA / "departments.csv",
    sep=";"
)

## Duplicados exactos

In [682]:
duplicados = (
    orders[
        orders.duplicated(
            keep=False
        )
    ]
    .sort_values(
        "order_id"
    )
)

print(len(duplicados))

duplicados[
    [
        "order_id",
        "order_dow",
        "order_hour_of_day"
    ]
].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


Eliminar duplicados:

In [683]:
orders = (
    orders
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)

print(
    orders.shape,
    orders["order_id"].is_unique
)

(478952, 6) True


In [684]:
print(
    "Antes:",
    round(
        orders.memory_usage(
            deep=True
        ).sum()/1e6,
        1
    ),
    "MB"
)

Antes: 23.0 MB


In [685]:
orders = orders.astype({
    "order_id":"int32",
    "user_id":"int32",
    "order_number":"int16",
    "order_dow":"int8",
    "order_hour_of_day":"int8",
    "days_since_prior_order":"Int16"
})

In [686]:
print(
    "Después:",
    round(
        orders.memory_usage(
            deep=True
        ).sum()/1e6,
        1
    ),
    "MB"
)

Después: 7.2 MB


In [687]:
nulos = (
    orders[
        "days_since_prior_order"
    ].isna()
)

primeros = (
    orders[
        "order_number"
    ] == 1
)

print(
    (nulos == primeros).all()
)

True


Los valores nulos de days_since_prior_order corresponden a primeros pedidos y deben conservarse.

In [688]:
products[
    products[
        "product_name"
    ].isna()
][
    ["aisle_id","department_id"]
].value_counts()

aisle_id  department_id
100       21               1258
Name: count, dtype: int64

In [689]:
products["product_name"] = (
    products["product_name"]
    .fillna("Unknown")
)

In [690]:
products[
    "product_name"
].isna().sum()

np.int64(0)

In [691]:
nombres = products["product_name"]

print(
    "Espacios dobles:",
    nombres.str.contains(
        "  ",
        regex=False
    ).sum()
)

print(
    "Espacio invisible:",
    nombres.str.contains(
        "\xa0",
        regex=False
    ).sum()
)

Espacios dobles: 403
Espacio invisible: 16


In [692]:
products["product_name"] = (
    products["product_name"]
    .str.replace(
        "\xa0",
        " ",
        regex=False
    )
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
    .str.title()
)

In [693]:
print(
    products[
        "product_name"
    ].str.contains(
        "  ",
        regex=False
    ).sum()
)

print(
    products[
        "product_name"
    ].str.contains(
        "\xa0",
        regex=False
    ).sum()
)

0
0


In [694]:
conocidos = products[
    products[
        "product_name"
    ] != "Unknown"
]

In [695]:
repetidos = conocidos[
    conocidos[
        "product_name"
    ].duplicated(
        keep=False
    )
]

In [696]:
print(
    conocidos[
        "product_name"
    ].duplicated()
    .sum()
)

141


In [697]:
print(
    conocidos["product_name"]
    .duplicated()
    .sum()
)

141


In [698]:
def porcentaje_nulos(df):
    return (
        df.isna()
        .mean()
        .mul(100)
        .round(2)
        .sort_values(ascending=False)
    )

In [699]:
def duplicados_explicitos(df):
    return df.duplicated().sum()

In [700]:
def memoria_mb(df):
    return round(
        df.memory_usage(
            deep=True
        ).sum() / 1e6,
        2
    )

In [701]:
porcentaje_nulos(orders)

days_since_prior_order    6.02
order_id                  0.00
user_id                   0.00
order_number              0.00
order_dow                 0.00
order_hour_of_day         0.00
dtype: float64

In [702]:
duplicados_explicitos(orders)

np.int64(0)

In [703]:
memoria_mb(orders)

np.float64(7.18)

Sección 9. Validación final de limpieza

In [704]:
assert orders.shape == (478952, 6)

assert (
    products["product_name"]
    .isna()
    .sum() == 0
)

assert (
    duplicados_explicitos(
        orders
    ) == 0
)

print("✅ Lección 2 validada")

✅ Lección 2 validada


# Laboratorio Guiado 2
## Construcción de un pipeline de limpieza

In [705]:
registro = []

In [706]:
def anotar(paso, detalle):
    registro.append({
        "paso": paso,
        "detalle": detalle
    })

In [707]:
anotar(
    "orders",
    "Duplicados eliminados"
)

anotar(
    "orders",
    "Tipos optimizados"
)

anotar(
    "products",
    "Nulos reemplazados por Unknown"
)

anotar(
    "products",
    "Texto estandarizado"
)

In [708]:
informe = pd.DataFrame(
    registro
)

informe

,paso,detalle
0,orders,Duplicados eliminados
1,orders,Tipos optimizados
2,products,Nulos reemplazados por Unknown
3,products,Texto estandarizado


In [709]:
Path("../data/processed").mkdir(
    parents=True,
    exist_ok=True
)

In [710]:
informe.to_csv(
    "../data/processed/cleaning_log.csv",
    index=False
)

In [711]:
assert len(informe) >= 4

print(
    "✅ Laboratorio Guiado 2 completado"
)

✅ Laboratorio Guiado 2 completado


In [712]:
# Reto 2
## Validación y análisis de calidad de datos

In [713]:
print("Duplicados exactos:")
print(order_prods.duplicated().sum())

print("\nPares repetidos order-product:")
print(
    order_prods.duplicated(
        subset=["order_id", "product_id"]
    ).sum()
)

print("\nValores reordered:")
print(
    sorted(
        order_prods["reordered"].unique()
    )
)

Duplicados exactos:
0

Pares repetidos order-product:
0

Valores reordered:
[np.int64(0), np.int64(1)]


### Interpretación

Se verificó la existencia de duplicados explícitos,
duplicados por combinación de pedido y producto,
y los valores presentes en la variable reordered.

### Interpretación

No se encontraron filas duplicadas en order_prods ni combinaciones repetidas de order_id y product_id.

Además, la variable reordered únicamente contiene los valores 0 y 1, lo que confirma que funciona como una variable binaria para indicar si un producto fue recomprado.

In [714]:
print(
    "Duplicados en aisles:",
    aisles["aisle"].duplicated().sum()
)

print(
    "Duplicados en departments:",
    departments["department"].duplicated().sum()
)
print(
    "Espacios en aisles:",
    (
        aisles["aisle"]
        != aisles["aisle"].str.strip()
    ).sum()
)

print(
    "Espacios en departments:",
    (
        departments["department"]
        != departments["department"].str.strip()
    ).sum()
)



Duplicados en aisles: 0
Duplicados en departments: 0
Espacios en aisles: 0
Espacios en departments: 0


### Interpretación

Las tablas aisles y departments no presentan nombres duplicados ni espacios adicionales al inicio o al final de los textos.

Por lo tanto, no requieren procesos adicionales de limpieza o estandarización.

In [715]:
order_prods["add_to_cart_order"].isna().sum()


np.int64(836)

In [716]:
pedidos_nulos = order_prods.loc[
    order_prods["add_to_cart_order"].isna(),
    "order_id"
]

In [717]:
pedidos_nulos.nunique()

70

In [718]:
productos_por_pedido = (
    order_prods
    .groupby("order_id")
    .size()
)

In [719]:
productos_por_pedido.loc[
    pedidos_nulos.unique()
].describe()

count     70.000000
mean      75.942857
std       12.898585
min       65.000000
25%       67.000000
50%       71.000000
75%       78.000000
max      127.000000
dtype: float64

In [720]:
print(
    "Nulos:",
    order_prods["add_to_cart_order"].isna().sum()
)

print(
    "Pedidos afectados:",
    pedidos_nulos.nunique()
)

productos_por_pedido.loc[
    pedidos_nulos.unique()
].describe()

Nulos: 836
Pedidos afectados: 70


count     70.000000
mean      75.942857
std       12.898585
min       65.000000
25%       67.000000
50%       71.000000
75%       78.000000
max      127.000000
dtype: float64

### Interpretación

Se identificaron 836 valores nulos en la columna add_to_cart_order.

Estos valores nulos aparecen distribuidos en 70 pedidos diferentes.

Los pedidos afectados contienen una gran cantidad de productos, con un mínimo de 65 productos por pedido, una mediana de 71 y un máximo de 127 productos.

Esto sugiere que los valores faltantes están asociados a pedidos excepcionalmente grandes, donde la posición exacta de algunos productos dentro del carrito no fue registrada durante la captura de los datos.

Por esta razón, no parece tratarse de un error aleatorio sino de una característica específica de pedidos con muchos artículos.

In [721]:
products_crudo = pd.read_csv(
    "../data/products.csv",
    sep=";"
)

In [722]:
comparacion = pd.DataFrame({
    "original": products_crudo["product_name"],
    "limpio": products["product_name"]
})

In [723]:
cambios = (
    comparacion["original"]
    .fillna("Unknown")
    != comparacion["limpio"]
).sum()

print("Nombres modificados:", cambios)

Nombres modificados: 6321


In [724]:
comparacion[
    comparacion["original"]
    .fillna("Unknown")
    != comparacion["limpio"]
].head(10)

,original,limpio
17,Pizza for One Suprema Frozen Pizza,Pizza For One Suprema Frozen Pizza
23,Tri-Vi-Sol® Vitamins A-C-and D Supplement Drop...,Tri-Vi-Sol® Vitamins A-C-And D Supplement Drop...
29,Three Cheese Ziti- Marinara with Meatballs,Three Cheese Ziti- Marinara With Meatballs
35,Traditional Lasagna with Meat Sauce Savory Ita...,Traditional Lasagna With Meat Sauce Savory Ita...
41,Biotin 1000 mcg,Biotin 1000 Mcg
45,Raisin Cinnamon Bagels 5 count,Raisin Cinnamon Bagels 5 Count
48,Vegetarian Grain Meat Sausages Italian - 4 CT,Vegetarian Grain Meat Sausages Italian - 4 Ct
63,Autumn Vegetable & Turkey Dinner with Lil' Bit...,Autumn Vegetable & Turkey Dinner With Lil' Bit...
68,Vanilla with Almond Milk Iced Coffee,Vanilla With Almond Milk Iced Coffee
73,Artisan Chick'n & Apple Sausage,Artisan Chick'N & Apple Sausage


### Interpretación

Se identificaron 836 valores nulos en la columna add_to_cart_order.

Estos valores nulos aparecen distribuidos en 70 pedidos diferentes.

Los pedidos afectados contienen una gran cantidad de productos, con un mínimo de 65 productos por pedido, una mediana de 71 y un máximo de 127 productos.

Esto sugiere que los valores faltantes están asociados a pedidos excepcionalmente grandes, donde la posición exacta de algunos productos dentro del carrito no fue registrada durante la captura de los datos.

Por esta razón, no parece tratarse de un error aleatorio sino de una característica específica de pedidos con muchos artículos.

### Interpretación

Se identificaron 6321 nombres de productos que cambiaron después del proceso de estandarización.

Los cambios corresponden principalmente a la normalización del formato de escritura, incluyendo corrección de mayúsculas y minúsculas, espacios innecesarios y consistencia en los nombres.

La estandarización mejora la calidad de los datos y facilita análisis posteriores basados en texto.

In [725]:
dias = {
    0: "Sunday",
    1: "Monday",
    2: "Tuesday",
    3: "Wednesday",
    4: "Thursday",
    5: "Friday",
    6: "Saturday"
}

day_names = orders["order_dow"].map(dias)

In [726]:
day_cat = pd.Categorical(
    day_names,
    categories=[
        "Sunday",
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday"
    ],
    ordered=True
)

In [727]:
print(
    "Texto:",
    day_names.memory_usage(deep=True),
    "bytes"
)

print(
    "Categoría:",
    day_cat.memory_usage(deep=True),
    "bytes"
)

Texto: 26835934 bytes
Categoría: 479645 bytes


In [728]:
day_cat

['Saturday', 'Friday', 'Friday', 'Sunday', 'Thursday', ..., 'Monday', 'Wednesday', 'Tuesday', 'Wednesday', 'Monday']
Length: 478952
Categories (7, str): ['Sunday' < 'Monday' < 'Tuesday' < 'Wednesday' < 'Thursday' < 'Friday' < 'Saturday']

### Interpretación

Se creó una variable categórica ordenada para representar los días de la semana.

La comparación de memoria muestra una reducción considerable del consumo, pasando de 26.835.934 bytes en formato texto a 479.645 bytes en formato categoría.

Esto ocurre porque las categorías almacenan una sola vez cada valor único y luego utilizan códigos internos para representarlos, lo que resulta especialmente eficiente cuando existen muchos valores repetidos.

Además, el orden explícito de las categorías facilita futuros análisis, agrupaciones y visualizaciones cronológicas.

Reto 2 - Ejercicio 6

La pregunta es:

¿Cuántas filas de order_prods corresponden a productos "Unknown" y en cuántos pedidos aparecen? ¿Los eliminarías?

In [729]:
unknown_ids = products.loc[
    products["product_name"] == "Unknown",
    "product_id"
]

unknown_rows = order_prods[
    order_prods["product_id"].isin(
        unknown_ids
    )
]
print(
    "Filas:",
    len(unknown_rows)
)
print(
    "Pedidos:",
    unknown_rows["order_id"].nunique()
)


Filas: 9547
Pedidos: 8254


### Interpretación

Se identificaron 9547 registros en order_prods asociados a productos con nombre "Unknown".

Estos registros aparecen distribuidos en 8254 pedidos diferentes, lo que indica que no se trata de casos aislados.

No sería recomendable eliminar estos registros, ya que representan compras reales realizadas por los usuarios. Aunque el nombre del producto no esté disponible, la información sigue siendo valiosa para análisis de pedidos, frecuencia de compra y comportamiento de los clientes.

Eliminar estos registros podría generar sesgos y afectar los resultados de futuros análisis.

Reto 2 - Ejercicio 7

En una copia de departments, reemplaza "missing" por pd.NA y explica por qué no conviene hacerlo en la tabla original antes de agrupar.

In [730]:
departments_copy = departments.copy()

departments_copy["department"] = (
    departments_copy["department"]
    .replace("missing", pd.NA)
)
departments_copy.isna().sum()

departments_copy[
    departments_copy["department"].isna()
]

,department_id,department
20,21,NaN


### Interpretación

Se realizó el reemplazo de "missing" por un valor nulo real (pd.NA) únicamente en una copia de la tabla.

No conviene modificar la tabla original antes de realizar agrupaciones porque muchos métodos de análisis excluyen automáticamente los valores nulos. Esto podría provocar que algunos registros desaparezcan de los resultados agregados y generar interpretaciones incorrectas.

Lección 3: Filtrado, ordenamiento y agrupación de datos

Sección 1. Filtrado con máscaras booleanas

In [731]:
es_primer_pedido = (
    orders["order_number"] == 1
)

print(
    es_primer_pedido.head(3)
)

primeros = orders[
    es_primer_pedido
]

print(
    len(primeros)
)

0    False
1    False
2    False
Name: order_number, dtype: bool
28817


In [732]:
es_fin_de_semana = (
    orders["order_dow"]
    .isin([0, 6])
)

es_manana = (
    orders["order_hour_of_day"]
    .between(6, 11)
)

print(
    len(
        orders[
            es_fin_de_semana &
            es_manana
        ]
    )
)

45599


Sección 2. Filtrado con query

In [733]:
nocturnos_30 = orders.query(
    "order_hour_of_day >= 22 and days_since_prior_order == 30"
)

print(len(nocturnos_30))

1449


Sección 3. Ordenamiento

In [734]:
print(
    orders.nlargest(
        3,
        "order_number"
    )[[
        "user_id",
        "order_number"
    ]]
)

historial = orders.sort_values(
    [
        "user_id",
        "order_number"
    ]
)

print(
    historial[
        [
            "user_id",
            "order_number"
        ]
    ].head(3)
)

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100
        user_id  order_number
237012        2             1
317136        2             4
337239        4             2


Sección 4. GroupBy

In [735]:
pedidos_por_hora = (
    orders
    .groupby("order_hour_of_day")[
        "order_id"
    ]
    .count()
)

print(
    pedidos_por_hora.idxmax(),
    pedidos_por_hora.max()
)

10 40578


Sección 5. Agrupación múltiple

In [736]:
resumen_dia = (
    orders
    .groupby("order_dow")
    .agg(
        pedidos=(
            "order_id",
            "count"
        ),
        clientes=(
            "user_id",
            "nunique"
        ),
        espera_media=(
            "days_since_prior_order",
            "mean"
        )
    )
    .round(1)
)

resumen_dia

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60897,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


Sección 6. Tabla dinámica

In [737]:
matriz = orders.pivot_table(
    index="order_hour_of_day",
    columns="order_dow",
    values="order_id",
    aggfunc="count"
)

print(matriz.shape)
matriz.loc[
    [8, 10, 14, 20]
]

(24, 7)


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


Sección 7. Transform

In [738]:
orders["pedidos_cliente"] = (
    orders
    .groupby("user_id")[
        "order_id"
    ]
    .transform("count")
)

frecuentes = orders[
    orders["pedidos_cliente"] >= 5
]

print(
    len(frecuentes),
    frecuentes["user_id"].nunique()
)

231946 30527


In [739]:
orders = orders.drop(
    columns=["pedidos_cliente"]
)

In [740]:
print(len(primeros))

print(len(nocturnos_30))

print(
    pedidos_por_hora.idxmax(),
    pedidos_por_hora.max()
)

print(
    len(frecuentes),
    frecuentes["user_id"].nunique()
)


28817
1449
10 40578
231946 30527


# Laboratorio Guiado 3
## Perfil de hábitos de compra

In [741]:
pedidos_por_cliente = (
    orders
    .groupby("user_id")["order_id"]
    .count()
)

print(
    len(pedidos_por_cliente)
)

pedidos_por_cliente.describe()

157437


count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

Paso 2. Segmentación de clientes

In [743]:
import numpy as np
segmento_cliente = pd.cut(
    pedidos_por_cliente,
    bins=[0, 1, 4, np.inf],
    labels=[
        "ocasional",
        "regular",
        "frecuente"
    ]
)

print(
    segmento_cliente
    .value_counts()
    .sort_index()
)

print(
    (
        segmento_cliente
        .value_counts(
            normalize=True
        )
        .sort_index()
        * 100
    ).round(1)
)


order_id
ocasional    55357
regular      71553
frecuente    30527
Name: count, dtype: int64
order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


In [744]:
orders["segmento"] = (
    orders["user_id"]
    .map(segmento_cliente)
)

orders[
    [
        "user_id",
        "order_number",
        "segmento"
    ]
].head()

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


Paso 4. Resumen por segmento

In [745]:
resumen_segmento = (
    orders
    .groupby(
        "segmento",
        observed=True
    )
    .agg(
        clientes=(
            "user_id",
            "nunique"
        ),
        pedidos=(
            "order_id",
            "count"
        ),
        espera_mediana=(
            "days_since_prior_order",
            "median"
        )
    )
)

resumen_segmento

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0


Paso 5. Franjas horarias

In [746]:
orders["franja"] = pd.cut(
    orders["order_hour_of_day"],
    bins=[-1, 5, 11, 17, 23],
    labels=[
        "madrugada",
        "mañana",
        "tarde",
        "noche"
    ]
)

Paso 6. Tabla cruzada

In [747]:
perfil = (
    pd.crosstab(
        orders["segmento"],
        orders["franja"],
        normalize="index"
    )
    .mul(100)
    .round(1)
)

perfil

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


Paso 7. Combinaciones más activas

In [748]:
(
    orders
    .groupby(
        ["order_dow", "franja"],
        observed=True
    )
    .size()
    .sort_values(
        ascending=False
    )
    .head(3)
)


order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

Paso 8. Limpiar columnas auxiliares

In [749]:
orders = orders.drop(
    columns=[
        "segmento",
        "franja"
    ]
)

print(
    orders.shape
)

(478952, 6)


In [758]:
segmento_cliente.value_counts().sort_index()

resumen_segmento

perfil



franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


In [759]:
print(orders.shape)

(478952, 6)


Reto 3
Segmentación y análisis de comportamiento

Ejercicio 1

La guía pide identificar qué porcentaje de pedidos corresponde a cada segmento.

In [763]:
pedidos_por_cliente = (
    orders
    .groupby("user_id")["order_id"]
    .count()
)

segmento_cliente = pd.cut(
    pedidos_por_cliente,
    bins=[0, 1, 4, float("inf")],
    labels=[
        "ocasional",
        "regular",
        "frecuente"
    ]
)
(
    segmento_cliente
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .sort_index()
)



order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64

Ejercicio 2

¿Cuál es el promedio de pedidos por usuario?

In [764]:
pedidos_por_cliente.mean()

np.float64(3.042181952145938)

Ejercicio 3

¿Cuál es el máximo número de pedidos hecho por un usuario?

In [765]:
pedidos_por_cliente.max()

np.int64(28)

Ejercicio 4

¿Cuántos usuarios realizaron un único pedido?

In [766]:
(
    pedidos_por_cliente == 1
).sum()

np.int64(55357)

Ejercicio 5

Usuarios con cinco o más pedidos

In [767]:
(
    pedidos_por_cliente >= 5
).sum()

np.int64(30527)

Ejercicio 6

¿Qué porcentaje representan los usuarios frecuentes?

In [768]:
(
    (pedidos_por_cliente >= 5).mean()
    * 100
).round(1)

np.float64(19.4)

Ejercicio 7

Validación final:

In [769]:
assert len(pedidos_por_cliente) == 157437

print("✅ Reto 3 completado")

✅ Reto 3 completado
